# GIK Real-Time Inference

## Imports

In [ ]:
import os
import sys
import yaml
import torch

from src.Constants.char_to_key import NUM_CLASSES, INDEX_TO_CHAR
from inference_preprocessing import preprocess, add_prev_char
from ml.models.gik_model import create_model

## Configuration

In [ ]:
PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Load training config from yaml file
TRAINING_CONFIG_PATH = os.path.join(PROJECT_ROOT, "train_config.yaml")
with open(TRAINING_CONFIG_PATH, "r", encoding="utf-8") as f:
    config_data = yaml.safe_load(f)

EXPERIMENT_MODE = config_data["experiment"]["mode"]
MODE_CONFIG = config_data["modes"][EXPERIMENT_MODE]

TRAINING_CONFIG = {
    **config_data["model"],
    **MODE_CONFIG    
}
TRAINING_CONFIG["output_logits"] = NUM_CLASSES

# Set inference config separately
INFERENCE_CONFIG = {
    "max_seq_length": 10,
    "normalize": True,
    "apply_filtering": True,
    "reduce_dim": True,
    "dim_red_method": "pca", # Set to None if reduce_dim == False
    "dims_ratio": 0.4 # Set to 0.0 if dims_red_method != "pca"
}

MODEL_PATH = os.path.join(PROJECT_ROOT, "best_model.pt")

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

## Receive Data, Load Pre-Trained ML Model, Run Inference

In [ ]:
prev_char = None

left = ...
right = ...

processed_data = preprocess(
    left_data = left,
    right_data = right,
    prev_char = prev_char,
    max_seq_length = INFERENCE_CONFIG["max_seq_length"],
    normalize = INFERENCE_CONFIG["normalize"],
    apply_filtering = INFERENCE_CONFIG["apply_filtering"],
    apply_dim_reduction = INFERENCE_CONFIG["reduce_dim"],
    dim_red_method = INFERENCE_CONFIG["dim_red_method"],
    dims_ratio = INFERENCE_CONFIG["dims_ratio"],
    root_dir = PROJECT_ROOT
)

input_dim = processed_data.shape[2] # Input dimension must match the data used to train the ML model

model = create_model(
    model_type = TRAINING_CONFIG["model_type"],
    hidden_dim_inner_model = TRAINING_CONFIG['hidden_dim_inner_model'],
    hidden_dim_classification_head = TRAINING_CONFIG['hidden_dim_classification_head'],
    no_layers_classification_head = TRAINING_CONFIG['num_layers'],
    dropout_inner_layers = TRAINING_CONFIG['dropout'],
    inner_model_kwargs = TRAINING_CONFIG['inner_model_prams'],
    output_logits = TRAINING_CONFIG['output_logits'],
    input_dim = input_dim
)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

with torch.no_grad():
    prediction = model.predict(processed_data).item()
    print(INDEX_TO_CHAR[prediction], end="")

    prev_char = prediction

## Proof of Concept

In [ ]:
import pandas as pd

DATA_DIR = config_data["data"]["data_dir"]
DATA_PATH = os.path.join(DATA_DIR, "Left_16.csv")
data = pd.read_csv(DATA_PATH, index_col=0).to_numpy()
nRows = (data.shape[0] // 10) * 10
data = data[:nRows].reshape(-1, 10, data.shape[1])

prev_char = None
for window in data:
    left = window
    right = None

    processed_data = preprocess(
        left_data = left,
        right_data = right,
        prev_char = prev_char,
        max_seq_length = INFERENCE_CONFIG["max_seq_length"],
        normalize = INFERENCE_CONFIG["normalize"],
        apply_filtering = INFERENCE_CONFIG["apply_filtering"],
        apply_dim_reduction = INFERENCE_CONFIG["reduce_dim"],
        dim_red_method = INFERENCE_CONFIG["dim_red_method"],
        dims_ratio = INFERENCE_CONFIG["dims_ratio"],
        root_dir = PROJECT_ROOT
    )

    input_dim = processed_data.shape[2]

    model = create_model(
        model_type = TRAINING_CONFIG["model_type"],
        hidden_dim_inner_model = TRAINING_CONFIG['hidden_dim_inner_model'],
        hidden_dim_classification_head = TRAINING_CONFIG['hidden_dim_classification_head'],
        no_layers_classification_head = TRAINING_CONFIG['num_layers'],
        dropout_inner_layers = TRAINING_CONFIG['dropout'],
        inner_model_kwargs = TRAINING_CONFIG['inner_model_prams'],
        output_logits = TRAINING_CONFIG['output_logits'],
        input_dim = input_dim
    )

    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()

    with torch.no_grad():
        prediction = model.predict(processed_data).item()
        print(INDEX_TO_CHAR[prediction], end="")

        prev_char = prediction

## Issues/To-Do
* Need to check if class index is mapped to one-hot encoding in order (i.e. whether using np.eye is valid)
* Need to check how to store previous prediction for regression
* Can consider having a pre-trained model and a set of PCA params for left hand only, right hand only and both hands just in case one hand malfunctions
* Only run inference if there is FSR data registered, regardless of whether one or two hands are active